In [1]:
import cv2
import numpy as np

In [2]:
img_a = cv2.imread("a.jpg")
img_b = cv2.imread("b.jpg")

In [3]:
gray_a = cv2.cvtColor(img_a, cv2.COLOR_BGR2GRAY)
gray_b = cv2.cvtColor(img_b, cv2.COLOR_BGR2GRAY)

In [4]:
sift = cv2.SIFT_create()
kp_a, des_a = sift.detectAndCompute(gray_a, None)
kp_b, des_b = sift.detectAndCompute(gray_b, None)

In [5]:
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des_a, des_b, k=2)

In [7]:
good_matches = []
for m, n in matches:
 if m.distance < 0.7 * n.distance:
  good_matches.append(m)

In [8]:
matched_keypoints_img = cv2.drawMatches(
img_a, kp_a, img_b, kp_b, good_matches, 
None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

In [9]:
src_pts = np.float32([kp_b[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2) 
dst_pts = np.float32([kp_a[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)

In [10]:
H, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

In [11]:
h_a, w_a = img_a.shape[:2]
h_b, w_b = img_b.shape[:2]

In [12]:
pts = np.float32([[0, 0], [0, h_b], [w_b, h_b], [w_b, 0]]).reshape(-1, 1, 2)
dst_corners = cv2.perspectiveTransform(pts, H)

In [13]:
all_corners = np.concatenate([dst_corners, np.float32([[0,0], [w_a,0], [w_a,h_a], 
[0,h_a]]).reshape(-1,1,2)], axis=0)
[x_min, y_min] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
[x_max, y_max] = np.int32(all_corners.max(axis=0).ravel() + 0.5)

In [16]:
translation_matrix = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]], dtype=np.float32)

stitched_img = cv2.warpPerspective(
img_b, 
translation_matrix @ H, 
(x_max - x_min, y_max - y_min))

In [17]:
stitched_img[-y_min : h_a - y_min, -x_min : w_a - x_min] = img_a

In [19]:
cv2.imshow('Matched SIFT Keypoints', matched_keypoints_img) 
cv2.imshow('Stitched Result', stitched_img) 
cv2.waitKey(0) 
cv2.destroyAllWindows()

In [20]:
cv2.imwrite("stitched_result.jpg", stitched_img)
print("拼接完成，结果已保存为 stitched_result.jpg")

拼接完成，结果已保存为 stitched_result.jpg
